<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_2_Pure_vs_Impure_Functions_in_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [2]:
import pandas as pd

# Replace with link of the your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

transactions_df = pd.read_csv(transactions, dtype={'user_id': 'Int64'})
transactions_df

,user_id,amount,transaction_date,status
0,1,250.50,1 03 2025,completed
1,2,0.00,2 03 2025,pending
2,1,175.75,1 03 2025,completed
3,3,89.99,3 03 2025,completed
4,2,-10.00,2 03 2025,failed
5,4,320.00,4 03 2025,completed
6,<NA>,150.00,1 03 2025,completed
7,1,200.00,5 03 2025,completed


In [3]:
# Pure – deterministic transformation
def double_numbers(numbers: list[int]) -> list[int]:
    return [x * 2 for x in numbers]

# data
lst_num = [2, 3, 4, 5]
# calling the function
double_numbers(lst_num)

[4, 6, 8, 10]

In [4]:
import csv

# impure function
def save_to_csv(data: list[int], path: str) -> None:
    """Save a list of integers to a CSV file at the given path."""
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(data)

## **Pure functions**

In [5]:
import pandas as pd

def drop_none_values(df: pd.DataFrame) -> pd.DataFrame:
    """Pure: filters out null values."""
    result = result.dropna(subset=['user_id', 'amount'])
    return result

def add_amount_usd(df: pd.DataFrame, rate: float) -> pd.DataFrame:
    """Pure: computes derived values."""
    result = df.copy() # Creates a copy
    result['amount_usd'] = result['amount'] * rate
    return result

## **Impure functions**

In [6]:
import pandas as pd

def load_csv(file_path: str) -> pd.DataFrame:
    """Impure: reads data from disk."""
    return pd.read_csv(file_path)

def save_results(df: pd.DataFrame, db_path: str) -> None:
    """Impure: writes data to a database."""
    conn = sqlite3.connect(db_path)
    try:
        df.to_sql(table, conn, if_exists="replace", index=False)
    finally:
        conn.close()

## **Not pure**

In [7]:
def fill_missing_amounts_inplace(df: pd.DataFrame) -> None:
    """Not Pure: modifies original DataFrame."""
    df.loc[df["amount"].isna(), "amount"] = 0.0
    return df

fill_missing_amounts_inplace(transactions_df)

,user_id,amount,transaction_date,status
0,1,250.50,1 03 2025,completed
1,2,0.00,2 03 2025,pending
2,1,175.75,1 03 2025,completed
3,3,89.99,3 03 2025,completed
4,2,-10.00,2 03 2025,failed
5,4,320.00,4 03 2025,completed
6,<NA>,150.00,1 03 2025,completed
7,1,200.00,5 03 2025,completed


In [8]:
from datetime import datetime

def add_amount_usd(df: pd.DataFrame, rate: float) -> pd.DataFrame:
    """Impure: Timestamp inside function."""
    result = df.copy()
    result['amount_usd'] = result['amount'] * rate
    result['Processed_time'] = datetime.now() # Different every run
    return result

## **Better Approach**

In [9]:
def add_amount_usd(df: pd.DataFrame, rate: float,
                   processed_time: datetime) -> pd.DataFrame:
    """Pure: adds the given timestamp to the DataFrame."""
    result = df.copy()
    result['amount_usd'] = result['amount'] * rate
    result['Processed_time'] = processed_time
    return result

## **Logging  side effect**

In [10]:
def drop_none_values(df: pd.DataFrame) -> pd.DataFrame:
    """Not Pure: Function also handles logging."""
    result = result.dropna(subset=['user_id', 'amount'])
    logger.info("Nulls dropped successfully") # Side effect
    return result

# **Challenge (code)**

In [11]:
from datetime import datetime

batch = [
    {"id": 1, "status": "pending"},
    {"id": 2, "status": "completed"},
    {"id": 3, "status": "pending"},
]

def send_slack_notification(record_id):
    print(f"Notification sent for record {record_id}")

def process_batch(batch):
    for record in batch:
        if record.get("status") == "pending":
            record["processed_at"] = datetime.now()
            record["status"] = "completed"
            send_slack_notification(record["id"])
    return batch

## **Solution**

In [12]:
import logging
from copy import deepcopy
from datetime import datetime
from typing import Dict, List, Tuple
from IPython.display import display

logger = logging.getLogger(__name__)

def is_pending(record: Dict) -> bool:
    """Check if a transaction record has a pending status."""
    return record.get("status") == "pending"

def mark_completed(record: Dict, processed_at: datetime) -> Dict:
    """Return a new deep-copied record with an updated completed status"""
    new_record = deepcopy(record)
    new_record["processed_at"] = processed_at
    new_record["status"] = "completed"
    return new_record

def transform_batch(batch: List[Dict],
                    processed_at: datetime) -> Tuple[List[Dict], List[int]]:
    """Separate pending rows, update their status"""
    transformed = []
    notification_ids = []

    for record in batch:
        if is_pending(record):
            transformed.append(mark_completed(record, processed_at))
            notification_ids.append(record["id"])
        else:
            transformed.append(deepcopy(record))
    return transformed, notification_ids

def send_slack_notification(record_id: int) -> None:
    """Simulate sending an outbound alert notification message."""
    logger.info(f"Notification sent for record {record_id}")

def process_batch(batch: List[Dict]) -> List[Dict]:
    """Orchestrate the batch processing cycle"""
    processed_at = datetime.now()

    transformed_batch, notification_ids = transform_batch(batch, processed_at)

    for record_id in notification_ids:
        send_slack_notification(record_id)
    return transformed_batch

# Execute pipeline
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    results = process_batch(batch)
    display(results)

INFO: Notification sent for record 1
INFO: Notification sent for record 3


[{'id': 1,
  'status': 'completed',
  'processed_at': datetime.datetime(2026, 11, 15, 19, 56, 26, 693669)},
 {'id': 2, 'status': 'completed'},
 {'id': 3,
  'status': 'completed',
  'processed_at': datetime.datetime(2026, 11, 15, 19, 56, 26, 693669)}]